# Correlation & Covariance
**Topic:** Descriptive Statistics

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Calculate** covariance and Pearson correlation between two variables
- **Explain** why correlation is preferable to covariance for comparing relationships across datasets
- **Interpret** a correlation value between −1 and +1 and recognize that correlation does not imply causation

> **Tip:** Start by moving the **correlation strength slider** from 0 to 1, watch how the scatter cloud changes from a circular blob into a tight diagonal line.

---
## How we got here

In *04–06* we described single variables: their center, spread, and shape. Now we take the next step and ask how two variables move **together**. This is the foundation of every regression model, every feature-selection algorithm, and every time we claim two things are related.

---
## Why this matters for data science

Correlation analysis is often the first step in exploratory data analysis. It drives feature selection (correlated features are redundant), flags data leakage (a feature too correlated with the target is suspicious), and underlies the Pearson correlation matrix that every pandas `.corr()` call produces. Principal Component Analysis (PCA) is essentially a way of finding the directions of maximum correlation in a high-dimensional dataset.

---
## Try it yourself

In [2]:
out = Output()

PAIRS = {
    "Height / Weight": {
        "x_label": "Height (cm)",
        "y_label": "Weight (kg)",
        "x_mu": 170.0, "x_sigma": 9.0,
        "y_mu": 70.0,  "y_sigma": 12.0,
        "default_r": 0.70,
        "default_noise": 1.0,
    },
    "Study Hours / Grade": {
        "x_label": "Study Hours (hrs/week)",
        "y_label": "Exam Grade (%)",
        "x_mu": 5.5,  "x_sigma": 2.2,
        "y_mu": 68.0, "y_sigma": 14.0,
        "default_r": 0.55,
        "default_noise": 1.4,
    },
    "Temperature / Ice Cream Sales": {
        "x_label": "Temperature (°C)",
        "y_label": "Ice Cream Sales (units/day)",
        "x_mu": 22.0, "x_sigma": 8.0,
        "y_mu": 280.0, "y_sigma": 90.0,
        "default_r": 0.80,
        "default_noise": 1.0,
    },
    "Age / Income": {
        "x_label": "Age (years)",
        "y_label": "Income ($k/year)",
        "x_mu": 42.0, "x_sigma": 10.0,
        "y_mu": 72.0, "y_sigma": 28.0,
        "default_r": 0.40,
        "default_noise": 1.8,
    },
    "Hours Slept / Errors Made": {
        "x_label": "Hours Slept",
        "y_label": "Errors Made (count)",
        "x_mu": 7.0,  "x_sigma": 1.2,
        "y_mu": 8.0,  "y_sigma": 4.0,
        "default_r": -0.60,
        "default_noise": 1.2,
    },
}

_INIT = PAIRS["Height / Weight"]

corr_slider = FloatSlider(
    value=_INIT["default_r"], min=-1.0, max=1.0, step=0.05,
    description="Correlation:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="440px"),
)
noise_slider = FloatSlider(
    value=_INIT["default_noise"], min=0.1, max=3.0, step=0.1,
    description="Noise:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="440px"),
)
noise_note = widgets.HTML(
    value=(
        '<div style="font-size:12px;color:#666;margin:-2px 0 8px 10px;line-height:1.55;">'
        '<b>Noise</b> scales the random variability that is unrelated to the relationship '
        'between the two variables — think of it as other factors, measurement error, and '
        'individual differences. At <b>1.0</b> the observed r matches the correlation slider. '
        'Above 1.0, extra scatter pulls r toward zero; below 1.0, the cloud tightens and r '
        'rises toward the slider value.'
        '</div>'
    )
)
pair_dropdown = Dropdown(
    options=list(PAIRS.keys()),
    value="Height / Weight",
    description="Variable Pair:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="440px"),
)
size_slider = IntSlider(
    value=150, min=50, max=500, step=50,
    description="Sample Size:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="440px"),
)

# ── Helpers ───────────────────────────────────────────────────────────────────

def _interp_text(r, x_label, y_label):
    abs_r = abs(r)
    x_name = x_label.split(" (")[0]
    y_name = y_label.split(" (")[0]
    if abs_r >= 0.7:
        strength = "Strong"
    elif abs_r >= 0.4:
        strength = "Moderate"
    elif abs_r >= 0.2:
        strength = "Weak"
    else:
        strength = "Very weak"
    direction = "positive" if r >= 0 else "negative"
    verb = "increases" if r >= 0 else "decreases"
    return (
        f"{strength} {direction} correlation — "
        f"as {x_name} increases, {y_name} tends to {verb}"
    )

# ── Render ────────────────────────────────────────────────────────────────────

def render(r_target, noise, pair_name, n):
    np.random.seed(42)
    p = PAIRS[pair_name]

    r_clamped = float(np.clip(r_target, -0.9999, 0.9999))
    cov_matrix = [[1.0, r_clamped], [r_clamped, 1.0]]
    x_std, y_base = np.random.multivariate_normal([0.0, 0.0], cov_matrix, n).T
    residual = y_base - r_clamped * x_std
    y_std = r_clamped * x_std + noise * residual

    x_vals = p["x_mu"] + p["x_sigma"] * x_std
    y_vals = p["y_mu"] + p["y_sigma"] * y_std

    r_val, _ = stats.pearsonr(x_vals, y_vals)
    r2_val   = r_val ** 2
    cov_val  = np.cov(x_vals, y_vals)[0, 1]

    slope, intercept, *_ = stats.linregress(x_vals, y_vals)
    x_line = np.linspace(x_vals.min(), x_vals.max(), 200)
    y_line = slope * x_line + intercept

    if abs(r_val) < 0.1:
        interp = (
            f"Near-zero correlation — covariance is {cov_val:.2f} but scale makes "
            f"comparison difficult; r normalizes this to {r_val:.3f}"
        )
        border_color = PALETTE["muted"]
    else:
        interp = _interp_text(r_val, p["x_label"], p["y_label"])
        border_color = PALETTE["primary"] if r_val >= 0 else PALETTE["secondary"]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x_vals, y=y_vals,
        mode="markers",
        marker=dict(color=PALETTE["primary"], size=6, opacity=0.55),
        name="Data points",
    ))
    fig.add_trace(go.Scatter(
        x=x_line, y=y_line,
        mode="lines",
        line=dict(color=PALETTE["secondary"], width=2.5),
        name="Regression line",
    ))

    fig_layout = base_layout(
        title=f"{pair_name}  |  r = {r_val:.3f},  r² = {r2_val:.3f}",
        xaxis_title=p["x_label"],
        yaxis_title=p["y_label"],
    )
    fig.update_layout(fig_layout)

    summary = (
        '<div style="background:#f8f9fa; border-left:4px solid ' + border_color + '; '
        'padding:14px 18px; margin-top:12px; border-radius:4px; '
        'font-size:14px; line-height:1.8;">'
        '<span style="font-weight:600;">Pearson r</span> = ' + f'{r_val:.3f}' +
        '&nbsp;&nbsp;<span style="font-weight:600;">r²</span> = ' + f'{r2_val:.3f}' +
        '&nbsp;&nbsp;|&nbsp;&nbsp;'
        '<span style="font-weight:600;">Covariance</span> = ' + f'{cov_val:.2f}' +
        '<br>' + interp + '</div>'
    )

    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))
        display(HTML(summary))

# ── Observers ─────────────────────────────────────────────────────────────────

_updating_controls = False

def _on_pair_change(change):
    global _updating_controls
    pair_name = change["new"]
    p = PAIRS[pair_name]
    _updating_controls = True
    corr_slider.value  = p["default_r"]
    noise_slider.value = p["default_noise"]
    _updating_controls = False
    render(corr_slider.value, noise_slider.value, pair_name, size_slider.value)

def on_change(change):
    if _updating_controls:
        return
    render(corr_slider.value, noise_slider.value, pair_dropdown.value, size_slider.value)

corr_slider.observe(on_change,      names="value")
noise_slider.observe(on_change,     names="value")
pair_dropdown.observe(_on_pair_change, names="value")
size_slider.observe(on_change,      names="value")

# ── Display ───────────────────────────────────────────────────────────────────

controls = VBox([corr_slider, noise_slider, noise_note, pair_dropdown, size_slider])
layout   = VBox([controls, out])

if not hasattr(out, "_initialized"):
    display(layout)
    out._initialized = True

render(corr_slider.value, noise_slider.value, pair_dropdown.value, size_slider.value)


---
## What's happening?

Covariance measures whether two variables tend to increase together or move in opposite directions. But its value depends on the units of measurement, comparing covariances across datasets is meaningless. Correlation solves this by **standardizing** the covariance into a unit-free number between −1 and +1.

| Measure | Symbol | What it measures |
|---------|--------|-----------------|
| Covariance | Cov(X,Y) | Direction of the linear relationship; unit-dependent |
| Pearson correlation | r | Strength and direction of linear relationship; unit-free |
| r² (coefficient of determination) | r² | Fraction of variance in Y explained by X |

$$\text{Cov}(X,Y) = \frac{1}{n-1}\sum_{i=1}^n (x_i - \bar{x})(y_i - \bar{y})$$

$$r = \frac{\text{Cov}(X,Y)}{s_X \cdot s_Y}$$

### Correlation is not causation

A high r tells you two variables move together, it says nothing about which one causes the other, or whether both are driven by a third hidden variable. Ice cream sales and drowning rates are positively correlated, because both are driven by summer heat.

Return to the widget and try the different variable pairs, even perfectly linear relationships require domain knowledge to interpret causally.

---
## Real-world example: Study hours and exam scores

The relationship between time spent studying and exam performance is one of the most studied correlations in education research. The data shows a positive relationship, but with meaningful scatter, other factors (sleep, prior knowledge, test anxiety) all add noise.

The chart below uses simulated data calibrated to findings from university student performance studies. Notice:

- **Notice:** The trend is clearly positive, more study hours generally predict higher scores, but individual points scatter widely around the regression line
- **Notice:** The relationship is not perfectly linear; returns diminish at very high study hours (students plateau near 100%)
- **Notice:** The r² value tells us what fraction of score variation is explained by study hours alone, a reminder that one variable rarely tells the whole story

> **Discussion question:** A student scoring 65% after studying 4 hours decides to study 8 hours for the next exam. Based on this chart, what score would you predict, and how confident should they be in that prediction?

In [3]:
np.random.seed(33)

# ── Study hours vs exam score — calibrated to education research literature ─────────────
n           = 120
study_hours = np.random.uniform(0.5, 12, n)
noise       = np.random.normal(0, 8, n)
# Diminishing returns: score = 40 + 5*hours - 0.2*hours^2 + noise
scores      = np.clip(40 + 5 * study_hours - 0.2 * study_hours**2 + noise, 0, 100)

r, p_value  = stats.pearsonr(study_hours, scores)

# Regression line
slope, intercept, *_ = stats.linregress(study_hours, scores)
x_line  = np.linspace(0.5, 12, 100)
y_line  = slope * x_line + intercept

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=study_hours, y=scores,
    mode="markers",
    marker=dict(color=PALETTE["primary"], size=7, opacity=0.65),
    name="Student",
))
fig.add_trace(go.Scatter(
    x=x_line, y=y_line,
    mode="lines",
    line=dict(color=PALETTE["secondary"], width=2.5),
    name=f"Regression line (r = {r:.2f}, r² = {r**2:.2f})",
))
layout = base_layout(
    title=f"Study Hours vs Exam Score  |  r = {r:.2f},  r² = {r**2:.2f}",
    xaxis_title="Hours Studied",
    yaxis_title="Exam Score (%)",
)
fig.update_layout(layout)
fig.show()

### Common correlation values and what they signal

| r value | Strength | Real-world example |
|---------|----------|-------------------|
| 0.90 – 1.00 | Very strong positive | A person's height measured in cm vs inches |
| 0.60 – 0.89 | Strong positive | Study hours vs exam score |
| 0.30 – 0.59 | Moderate positive | Exercise frequency vs resting heart rate (inverse) |
| −0.10 – 0.10 | Weak / negligible | Shoe size vs IQ |
| −0.60 – −0.89 | Strong negative | Price of a product vs quantity demanded |
| −1.00 | Perfect negative | Miles driven vs remaining fuel (constant speed) |

---
## When Pearson fails: Spearman correlation

Pearson r measures **linear** relationships. If the true relationship is monotonic but curved (more study hours always helps, but the gains diminish), Pearson underestimates the true association. **Spearman's rank correlation** solves this by converting both variables to ranks first, then computing Pearson r on the ranks.

| Property | Pearson r | Spearman ρ |
|----------|-----------|------------|
| Measures | Linear association | Monotonic association |
| Sensitive to outliers | Yes | Less so (ranks compress extremes) |
| Assumes | Continuous, approx. normal | Only that the relationship is monotonic |
| Best for | Normally distributed data | Ordinal data, skewed data, non-linear relationships |

**Rule of thumb:** plot the data first. If the scatter plot looks curved but consistently directional, prefer Spearman.

In [4]:
np.random.seed(17)

# Non-linear but monotonic relationship: income vs years of experience
# Income grows quickly early in career, then levels off
experience = np.random.uniform(0, 35, 120)
income = 35 + 60 * (1 - np.exp(-experience / 8)) + np.random.normal(0, 6, 120)
income = np.clip(income, 20, 120)

r_pearson, _  = stats.pearsonr(experience, income)
r_spearman, _ = stats.spearmanr(experience, income)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=experience, y=income,
    mode="markers",
    marker=dict(color=PALETTE["primary"], size=6, opacity=0.60),
    name="Individual",
))

# Fit a smoothing curve (lowess-style using sorted values)
sort_idx = np.argsort(experience)
x_smooth = experience[sort_idx]
y_true   = 35 + 60 * (1 - np.exp(-x_smooth / 8))  # noiseless curve
fig.add_trace(go.Scatter(
    x=x_smooth, y=y_true,
    mode="lines",
    line=dict(color=PALETTE["accent"], width=2.5),
    name="True monotonic trend",
))

# Pearson (linear) fit
slope, intercept, *_ = stats.linregress(experience, income)
x_line = np.linspace(0, 35, 200)
fig.add_trace(go.Scatter(
    x=x_line, y=slope * x_line + intercept,
    mode="lines",
    line=dict(color=PALETTE["secondary"], width=2, dash="dash"),
    name=f"Pearson linear fit (r = {r_pearson:.2f})",
))

layout = base_layout(
    title=(
        f"Years of Experience vs Income  |  "
        f"Pearson r = {r_pearson:.2f}  vs  Spearman ρ = {r_spearman:.2f}"
    ),
    xaxis_title="Years of Experience",
    yaxis_title="Annual Income ($k)",
)
layout.update(showlegend=True)
fig.update_layout(layout)
fig.show()

---
## Key takeaway

> **Correlation standardizes the direction and strength of a linear relationship into a single number between −1 and +1; a high correlation tells you two variables move together, but never why.**

---
*Next up: Data Visualization for Statistics — how to choose the right chart to reveal the patterns we have been calculating*